# 04 Continual Learning

Purpose: Rebuild an isolated ChromaDB at chroma_db_continual from scratch, then evaluate local DNDS as the DB grows from 10% to 100%.

Inputs: dataset/CVPR_2024_dataset_Train/, dataset/CVPR_2024_dataset_Test/, dataset_text/train.csv, dataset_text/test.csv.

Outputs: results/phase2/continual_learning_results.json, figures/phase2/continual_learning_curve.png.

In [ ]:
import math
import shutil
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy as np
from PIL import Image
from tqdm.auto import tqdm

from src.phase2.config import get_phase2_config
from src.phase2.data_utils import build_records_from_csv
from src.phase2.db_client import (
    get_image_collection,
    get_persistent_client,
    get_text_collection,
)
from src.phase2.evaluation import evaluate_variant, save_results
from src.phase2.scoring import local_dnds
from src.phase2.visualization import plot_continual_learning_curve

CONFIG = get_phase2_config({"continual_db_path": "./chroma_db_continual"})

REPO_ROOT = Path("../..").resolve()
TRAIN_DIR = REPO_ROOT / "dataset" / "CVPR_2024_dataset_Train"
TEST_DIR = REPO_ROOT / "dataset" / "CVPR_2024_dataset_Test"
TRAIN_CSV = REPO_ROOT / "dataset_text" / "train.csv"
TEST_CSV = REPO_ROOT / "dataset_text" / "test.csv"
RESULTS_PATH = REPO_ROOT / "results" / "phase2" / "continual_learning_results.json"
FIG_PATH = REPO_ROOT / "figures" / "phase2" / "continual_learning_curve.png"

for required in [TRAIN_DIR, TEST_DIR, TRAIN_CSV, TEST_CSV]:
    if not required.exists():
        raise FileNotFoundError(
            f"Missing required input for continual learning notebook: {required}"
        )

train_records, train_missing_examples, train_total_rows = build_records_from_csv(
    csv_path=TRAIN_CSV,
    split_dir=TRAIN_DIR,
    text_column="text",
    label_column="label",
    text_key="text",
)
test_samples, test_missing_examples, test_total_rows = build_records_from_csv(
    csv_path=TEST_CSV,
    split_dir=TEST_DIR,
    text_column="text",
    label_column="label",
    text_key="text",
)

if train_missing_examples:
    print("Skipped train rows with missing image files (up to 10 shown):")
    for item in train_missing_examples:
        print(f"  - {item}")

if test_missing_examples:
    print("Skipped test rows with missing image files (up to 10 shown):")
    for item in test_missing_examples:
        print(f"  - {item}")

if not train_records:
    raise RuntimeError("No train records found for continual-learning simulation.")
if not test_samples:
    raise RuntimeError("No test records found for continual-learning evaluation.")

print(f"Train records used: {len(train_records)} / {train_total_rows}")
print(f"Test samples used: {len(test_samples)} / {test_total_rows}")

In [ ]:
continual_db_dir = REPO_ROOT / "chroma_db_continual"
if continual_db_dir.exists():
    shutil.rmtree(continual_db_dir)

try:
    client = get_persistent_client(str(continual_db_dir))
    image_collection = get_image_collection(client)
    text_collection = get_text_collection(client)
except Exception as exc:
    raise RuntimeError(f"Failed to initialize isolated continual DB: {exc}") from exc


def add_records_to_db(records: list[dict], start_index: int) -> None:
    batch_size = CONFIG["batch_size"]
    for start in range(0, len(records), batch_size):
        batch = records[start : start + batch_size]
        ids_img, ids_txt, images, docs, metadatas = [], [], [], [], []

        for offset, record in enumerate(batch):
            global_idx = start_index + start + offset
            image_np = np.array(
                Image.open(record["image_path"]).convert("RGB"), dtype=np.uint8
            )
            ids_img.append(f"img_{global_idx}")
            ids_txt.append(f"txt_{global_idx}")
            images.append(image_np)
            docs.append(record["text"])
            metadatas.append({"label": record["label"], "filename": record["text"]})

        try:
            image_collection.add(ids=ids_img, images=images, metadatas=metadatas)
            text_collection.add(ids=ids_txt, documents=docs, metadatas=metadatas)
        except Exception as exc:
            raise RuntimeError(
                f"Failed adding records to continual DB at start={start_index + start}: {exc}"
            ) from exc


results = {
    "db_size_percent": [],
    "macro_f1": [],
    "steps": {},
}

added_count = 0
total_train = len(train_records)
for pct in tqdm(range(10, 101, 10), desc="Continual growth steps"):
    target_count = math.ceil(total_train * (pct / 100.0))
    target_count = min(target_count, total_train)

    if target_count > added_count:
        new_slice = train_records[added_count:target_count]
        add_records_to_db(new_slice, start_index=added_count)
        added_count = target_count

    image_count = image_collection.count()
    text_count = text_collection.count()
    if image_count != added_count or text_count != added_count:
        raise AssertionError(
            f"Collection counts mismatch at {pct}%: image={image_count}, text={text_count}, expected={added_count}"
        )

    metrics = evaluate_variant(
        local_dnds, test_samples, image_collection, text_collection, CONFIG
    )
    results["db_size_percent"].append(pct)
    results["macro_f1"].append(metrics["macro_f1"])
    results["steps"][str(pct)] = {
        "db_count": added_count,
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
        "weighted_f1": metrics["weighted_f1"],
        "inference_time_ms": metrics.get("inference_time_ms", 0.0),
    }

if image_collection.count() != total_train or text_collection.count() != total_train:
    raise AssertionError(
        f"Final 100% step must contain the complete training set. "
        f"image={image_collection.count()}, text={text_collection.count()}, expected={total_train}"
    )

In [ ]:
save_results(results, str(RESULTS_PATH))
plot_continual_learning_curve(results, str(FIG_PATH))

print("Continual learning experiment complete.")
print(f"Results saved to: {RESULTS_PATH}")
print(f"Figure saved to: {FIG_PATH}")
results